# Final Project: Peercoin Implementation
## Noah Burnham
## CS 3120: Secure Distributed Computation

In [3]:
# Imports and definitions
import numpy as np
import hashlib
from nacl.signing import SigningKey
from dataclasses import dataclass
from typing import List

In [9]:
DIFFICULTY = int(2**(32 * 8)/10000)

def bytes_value(v):
    return bytes(str(v), encoding='utf-8')

def hash_value(v):
    return hashlib.sha256(bytes_value(v)).hexdigest()

def get_blockchain_height(blockchain):
    height = 0
    curr = blockchain
    while curr is not None:
        height += 1
        curr = curr.prev.pointer
    return height

@dataclass
class HashPointer:
    hash: bytes
    pointer: any

# Transactions have inputs and outputs
@dataclass
class Transaction:
    inputs: list
    outputs: list

@dataclass
class Input:
    previous_tx: Transaction
    index: int
    public_key: SigningKey

# Signed inputs are signed by public keys
@dataclass
class SignedInput:
    input: Input
    signed_input: bytes

@dataclass
class Output:
    public_key_hash: hash
    value: int
    block_height: int = 0

# Each block has a list of transactions
@dataclass
class Block:
    transactions: list
    prev: HashPointer
    nonce: int
    pubkey_hash: hash
    block_height: int
    difficulty: int
    
    def __repr__(self):
        return f'\nBlock(\n transaction: {self.transactions},\n nonce: {self.nonce},\n prev: {self.prev}, \n block height: {self.block_height})'


In [10]:
def compute_coin_age(output, curr_height):
    age = curr_height - output.block_height
    return output.value * age

def get_total_stake(outputs, curr_height):
    total_stake = 0
    for o in outputs:
        total_stake += compute_coin_age(o, curr_height)
    return total_stake

def adjust_difficulty(stake):
    if stake > 0:
        scaled_difficulty = DIFFICULTY * (1 + stake / 100)
        return int(scaled_difficulty)
    return DIFFICULTY

In [11]:
def add_block(transactions, blockchain, nonce, pubkey_hash, difficulty):
    prev_hash = hash_value(blockchain)
    prev = HashPointer(prev_hash, blockchain)
    curr_height = get_blockchain_height(blockchain)
    for tx in transactions:
        for output in tx.outputs:
            if output.block_height == 0:
                output.block_height = curr_height + 1
    new_block = Block(transactions, prev, nonce, pubkey_hash, curr_height+1, difficulty)
    return new_block, hash_value(new_block)

def mine_for_block(transactions, pubkey_hash, blockchain, stake):
    # stake is a list of outputs
    curr_height = get_blockchain_height(blockchain)
    total_stake = get_total_stake(stake, curr_height)
    difficulty = adjust_difficulty(total_stake)    
    nonce = 0
    while True:
        nonce = nonce + 1
        new_block, final_hash = add_block(transactions, blockchain, nonce, pubkey_hash, difficulty)
        if int(final_hash, 16) <= difficulty:
            # reset the coin-ages once a block is mined
            for o in stake:
                o.block_height = curr_height + 1
            print("Found a block! Nonce: ", nonce)
            return new_block, final_hash

def check_blockchain(blockchain, expected_hash):
    curr = blockchain
    if hash_value(blockchain) != expected_hash:
        raise Exception("the blockchain does not match the expected hash.")
    while curr is not None:
        # check that the difficulty is satisfied
        if int(hash_value(curr), 16) > curr.difficulty:
            raise Exception("The difficulty requirement is not satisfied.")
        # check each transaction to make sure that the sender is allowed to use that money
        for tx in curr.transactions:
            for signed_input in tx.inputs:
                inp = signed_input.input
                if inp.previous_tx == 'COINBASE':
                    continue
                try:
                    # check the signature
                    inp.public_key.verify(bytes_value(inp), signed_input.signed_input.signature)
                except Exception:
                    raise Exception("invalid transaction -- sender tried using money that doesn't belong to them")
        curr = curr.prev.pointer
    return True